In [1]:
# ── Cell 1: Install (then restart runtime) ────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "-q",
                "gradio", "captum", "numpy==1.26.4"], check=True)
print("✅ Done. RESTART RUNTIME now, then run from Cell 2.")


✅ Done. RESTART RUNTIME now, then run from Cell 2.


In [1]:
# ══════════════════════════════════════════════════════════════
# ── Cell 2: Mount & Imports (run after restart) ───────────────
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import LayerIntegratedGradients

print("Gradio version:", gr.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


Mounted at /content/drive
Gradio version: 5.50.0
GPU: Tesla T4


In [2]:
# ── Cell 3: Paths & Config ────────────────────────────────────────────────────
DATA_DIR       = "/content/drive/MyDrive/disaster_project/disaster_data"
MODELS_DIR     = "/content/drive/MyDrive/disaster_project/disaster_models"
CONFIDENCE_DIR = "/content/drive/MyDrive/disaster_project/confidence_results"
MODEL_PATH     = f"{MODELS_DIR}/roberta/best_model"

with open(f"{DATA_DIR}/label_mapping.json") as f:
    mapping = json.load(f)

label2id    = mapping["label2id"]
id2label    = {int(k): v for k, v in mapping["id2label"].items()}
NUM_LABELS  = len(label2id)
CLASS_NAMES = [id2label[i] for i in range(NUM_LABELS)]

OPTIMAL_THRESHOLD = float(np.load(f"{CONFIDENCE_DIR}/optimal_threshold.npy"))
print(f"Confidence threshold: {OPTIMAL_THRESHOLD}")

CLASS_ICONS = {
    "caution_and_advice":                     "⚠️",
    "displaced_people_and_evacuations":        "🚶",
    "infrastructure_and_utility_damage":       "🏚️",
    "injured_or_dead_people":                  "🏥",
    "missing_or_found_people":                 "🔍",
    "not_humanitarian":                        "🚫",
    "other_relevant_information":              "ℹ️",
    "requests_or_urgent_needs":                "🆘",
    "rescue_volunteering_or_donation_effort":  "🤝",
    "sympathy_and_support":                    "💙",
}

CLASS_DESCRIPTIONS = {
    "caution_and_advice":                     "Official warnings and safety instructions",
    "displaced_people_and_evacuations":        "People forced to leave their homes",
    "infrastructure_and_utility_damage":       "Damage to roads, buildings, utilities",
    "injured_or_dead_people":                  "Casualties and injuries reported",
    "missing_or_found_people":                 "Missing persons or survivors found",
    "not_humanitarian":                        "Not related to humanitarian response",
    "other_relevant_information":              "Relevant but uncategorised information",
    "requests_or_urgent_needs":                "Urgent requests for help or supplies",
    "rescue_volunteering_or_donation_effort":  "Rescue operations and donations",
    "sympathy_and_support":                    "Emotional support and condolences",
}


Confidence threshold: 0.74


In [3]:
# ── Cell 4: Load Model ────────────────────────────────────────────────────────
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)
model.to(device)
model.eval()

lig = LayerIntegratedGradients(
    lambda input_ids, attn_mask: model(
        input_ids=input_ids, attention_mask=attn_mask
    ).logits,
    model.roberta.embeddings.word_embeddings,
)

print("✅ RoBERTa loaded and ready.")

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

✅ RoBERTa loaded and ready.


In [4]:
# ── Cell 5: Inference & Attribution Functions ─────────────────────────────────
def classify_tweet(text):
    enc = tokenizer(text, return_tensors="pt",
                    truncation=True, max_length=128, padding="max_length")
    input_ids      = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs  = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
    pred_class = int(np.argmax(probs))
    confidence = float(probs[pred_class])
    return probs, pred_class, confidence


def get_attributions(text, pred_class, n_steps=50):
    enc = tokenizer(text, return_tensors="pt",
                    truncation=True, max_length=128, padding="max_length")
    input_ids      = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)
    baseline_ids   = torch.zeros_like(input_ids)
    baseline_ids[:, 0]  = tokenizer.cls_token_id
    baseline_ids[:, -1] = tokenizer.sep_token_id

    attributions, _ = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attention_mask,),
        target=pred_class,
        n_steps=n_steps,
        return_convergence_delta=True,
    )
    attr_scores = attributions.sum(dim=-1).squeeze(0).detach().cpu().numpy()
    seq_len     = attention_mask.squeeze(0).sum().item()
    tokens      = tokenizer.convert_ids_to_tokens(
                      input_ids.squeeze(0).cpu().numpy())[:seq_len]
    attr_scores = attr_scores[:seq_len]
    max_abs     = np.abs(attr_scores).max()
    if max_abs > 0:
        attr_scores = attr_scores / max_abs
    return tokens, attr_scores


In [5]:
# ── Cell 6: Output HTML Builders ──────────────────────────────────────────────
def build_prediction_html(pred_class, confidence, threshold):
    is_uncertain = confidence < threshold
    icon         = CLASS_ICONS.get(CLASS_NAMES[pred_class], "•")
    label        = CLASS_NAMES[pred_class].replace("_", " ").title()
    desc         = CLASS_DESCRIPTIONS.get(CLASS_NAMES[pred_class], "")

    if is_uncertain:
        badge = f"""
        <div style="background:rgba(255,85,85,0.12);border:1px solid rgba(255,85,85,0.4);
                    border-radius:8px;padding:10px 14px;margin-top:14px;
                    font-family:'Space Mono',monospace;font-size:12px;
                    font-weight:700;color:#ff5555;display:flex;align-items:center;gap:10px;">
            ⚡ LOW CONFIDENCE — prediction may be unreliable
            <span style="font-weight:400;font-size:11px;opacity:0.8;">
                {confidence:.1%} is below threshold {threshold:.1%}
            </span>
        </div>"""
    else:
        badge = f"""
        <div style="background:rgba(0,229,160,0.08);border:1px solid rgba(0,229,160,0.3);
                    border-radius:8px;padding:10px 14px;margin-top:14px;
                    font-family:'Space Mono',monospace;font-size:12px;
                    font-weight:700;color:#00e5a0;display:flex;align-items:center;gap:10px;">
            ✅ HIGH CONFIDENCE — reliable prediction
            <span style="font-weight:400;font-size:11px;opacity:0.8;">
                {confidence:.1%} exceeds threshold {threshold:.1%}
            </span>
        </div>"""

    return f"""
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@400;600;700&display=swap" rel="stylesheet">
<div style="background:#0d1117;border-radius:12px;padding:20px;border:1px solid #21262d;">
    <div style="display:flex;align-items:center;gap:14px;">
        <div style="font-size:38px;">{icon}</div>
        <div style="flex:1;">
            <div style="font-family:'Space Mono',monospace;font-size:20px;
                        font-weight:700;color:#00e5a0;">{label}</div>
            <div style="font-size:13px;color:#8b949e;margin-top:3px;">{desc}</div>
        </div>
        <div style="text-align:center;min-width:80px;">
            <div style="font-family:'Space Mono',monospace;font-size:30px;
                        font-weight:700;color:#00e5a0;">{confidence:.0%}</div>
            <div style="font-size:10px;color:#6b7a8a;font-family:'Space Mono',monospace;
                        letter-spacing:1px;text-transform:uppercase;">Confidence</div>
        </div>
    </div>
    {badge}
</div>"""


def build_attribution_html(tokens, attributions):
    skip = {tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token}
    token_html = ""
    for tok, score in zip(tokens, attributions):
        if tok in skip:
            continue
        display_tok = tok.replace("Ġ", " ").replace("Ċ", " ")
        if score > 0:
            alpha = min(0.15 + abs(score) * 0.75, 0.92)
            bg    = f"rgba(0,210,120,{alpha:.2f})"
        else:
            alpha = min(0.15 + abs(score) * 0.75, 0.92)
            bg    = f"rgba(255,80,80,{alpha:.2f})"
        token_html += (
            f'<span style="background:{bg};padding:3px 3px;margin:1px 0;'
            f'border-radius:4px;font-size:15px;line-height:2.4;cursor:default;'
            f'font-family:DM Sans,sans-serif;" title="score: {score:.3f}">'
            f'{display_tok}</span>'
        )

    return f"""
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@400;600&display=swap" rel="stylesheet">
<div style="background:#0d1117;border-radius:12px;padding:20px;border:1px solid #21262d;">
    <div style="font-family:'Space Mono',monospace;font-size:10px;color:#6b7a8a;
                letter-spacing:2px;text-transform:uppercase;margin-bottom:14px;
                display:flex;justify-content:space-between;">
        <span>Integrated Gradients — Token Attribution</span>
        <span>hover to see score</span>
    </div>
    <div style="line-height:2.6;word-wrap:break-word;padding:4px 0;">
        {token_html}
    </div>
    <div style="display:flex;gap:20px;margin-top:14px;padding-top:12px;
                border-top:1px solid #21262d;flex-wrap:wrap;">
        <div style="display:flex;align-items:center;gap:7px;font-size:12px;
                    color:#8b949e;font-family:'Space Mono',monospace;">
            <div style="width:12px;height:12px;border-radius:3px;
                        background:rgba(0,210,120,0.75);"></div>
            Supports prediction
        </div>
        <div style="display:flex;align-items:center;gap:7px;font-size:12px;
                    color:#8b949e;font-family:'Space Mono',monospace;">
            <div style="width:12px;height:12px;border-radius:3px;
                        background:rgba(255,80,80,0.75);"></div>
            Opposes prediction
        </div>
        <div style="margin-left:auto;font-size:10px;color:#3a4a5a;
                    font-family:'Space Mono',monospace;">
            Darker = stronger attribution
        </div>
    </div>
</div>"""


def build_probs_html(probs, pred_class):
    sorted_cls = np.argsort(probs)[::-1]
    rows = ""
    for c in sorted_cls:
        pct     = probs[c] * 100
        is_top  = (c == pred_class)
        bar_col = "#00e5a0" if is_top else "#2a3a4a"
        txt_col = "#00e5a0" if is_top else "#6b7a8a"
        glow    = "box-shadow:0 0 8px rgba(0,229,160,0.4);" if is_top else ""
        icon    = CLASS_ICONS.get(CLASS_NAMES[c], "•")
        lbl     = CLASS_NAMES[c].replace("_", " ")
        fw      = "600" if is_top else "400"
        rows += f"""
        <div style="display:grid;grid-template-columns:250px 1fr 52px;
                    align-items:center;gap:10px;margin-bottom:8px;">
            <div style="font-size:12px;color:{txt_col};font-weight:{fw};
                        font-family:'DM Sans',sans-serif;white-space:nowrap;
                        overflow:hidden;text-overflow:ellipsis;">
                {icon} {lbl}
            </div>
            <div style="background:#1c2330;border-radius:4px;height:7px;overflow:hidden;">
                <div style="width:{pct:.1f}%;height:100%;background:{bar_col};
                            border-radius:4px;{glow}min-width:2px;"></div>
            </div>
            <div style="font-family:'Space Mono',monospace;font-size:11px;
                        color:{txt_col};text-align:right;">{pct:.1f}%</div>
        </div>"""

    return f"""
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@400;600&display=swap" rel="stylesheet">
<div style="background:#0d1117;border-radius:12px;padding:20px;border:1px solid #21262d;">
    <div style="font-family:'Space Mono',monospace;font-size:10px;color:#6b7a8a;
                letter-spacing:2px;text-transform:uppercase;margin-bottom:14px;">
        Class Probability Distribution
    </div>
    {rows}
</div>"""


In [6]:
# ── Cell 7: Main Gradio Prediction Function ───────────────────────────────────
def predict(tweet_text):
    if not tweet_text or not tweet_text.strip():
        empty = "<div style='color:#6b7a8a;font-family:monospace;padding:20px;'>Enter a tweet above and click Classify.</div>"
        return empty, empty, empty

    probs, pred_class, confidence = classify_tweet(tweet_text)
    tokens, attributions          = get_attributions(tweet_text, pred_class)

    pred_html  = build_prediction_html(pred_class, confidence, OPTIMAL_THRESHOLD)
    attr_html  = build_attribution_html(tokens, attributions)
    probs_html = build_probs_html(probs, pred_class)

    return pred_html, attr_html, probs_html


In [7]:
# ── Cell 8: Build & Launch Gradio App ────────────────────────────────────────
EXAMPLE_TWEETS = [
    "Massive flooding in Kerala, hundreds of families displaced. Roads submerged. Please send help.",
    "Power lines down across three districts after the earthquake. No electricity for 48 hours.",
    "Red Cross accepting food, water and medicine donations at the community center on 5th street.",
    "Thoughts and prayers to all the families affected by this terrible disaster.",
    "Breaking: 12 confirmed dead after Category 4 hurricane makes landfall near coastal towns.",
    "Government urges residents to stock up on water and avoid low-lying areas before the storm.",
    "Search teams looking for 3 missing children after flash floods swept through the village.",
    "Volunteers needed urgently at the relief camp. Please bring blankets and dry food.",
]

CSS = """
body, .gradio-container {
    background: #0d1117 !important;
    font-family: 'DM Sans', sans-serif !important;
}
.gr-button-primary {
    background: linear-gradient(135deg, #00e5a0, #0070f3) !important;
    border: none !important;
    color: #0d1117 !important;
    font-weight: 700 !important;
    font-family: 'Space Mono', monospace !important;
    letter-spacing: 1px !important;
    font-size: 13px !important;
}
.gr-button-primary:hover {
    opacity: 0.85 !important;
    transform: translateY(-1px) !important;
}
textarea, input {
    background: #161b22 !important;
    color: #e6edf3 !important;
    border: 1px solid #30363d !important;
    border-radius: 8px !important;
    font-size: 15px !important;
    font-family: 'DM Sans', sans-serif !important;
}
label span {
    color: #8b949e !important;
    font-family: 'Space Mono', monospace !important;
    font-size: 11px !important;
    letter-spacing: 2px !important;
    text-transform: uppercase !important;
}
.gr-examples-table {
    background: #161b22 !important;
    border: 1px solid #30363d !important;
    border-radius: 8px !important;
}
footer { display: none !important; }
"""

with gr.Blocks(
    css=CSS,
    theme=gr.themes.Base(
        primary_hue="emerald",
        neutral_hue="slate",
    ),
    title="Disaster Tweet Classifier",
) as demo:

    gr.HTML("""
    <link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;600;700&display=swap" rel="stylesheet">
    <div style="text-align:center;padding:28px 0 20px;border-bottom:1px solid #21262d;margin-bottom:24px;">
        <div style="font-size:32px;margin-bottom:8px;">⚡</div>
        <div style="font-family:'Space Mono',monospace;font-size:22px;font-weight:700;
                    color:#e6edf3;letter-spacing:-0.5px;">DISASTER TWEET CLASSIFIER</div>
        <div style="font-size:13px;color:#6b7a8a;margin-top:6px;
                    font-family:'DM Sans',sans-serif;">
            Humanitarian Information Categorisation System &nbsp;·&nbsp;
            RoBERTa + Integrated Gradients &nbsp;·&nbsp;
            10 Categories &nbsp;·&nbsp;
            Confidence Threshold: 0.74
        </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=2):
            tweet_input = gr.Textbox(
                label="Tweet Text",
                placeholder="Paste or type a disaster-related tweet here...",
                lines=4,
                max_lines=8,
            )
            classify_btn = gr.Button(
                "⚡  CLASSIFY TWEET",
                variant="primary",
                size="lg",
            )
            gr.Examples(
                examples=[[t] for t in EXAMPLE_TWEETS],
                inputs=tweet_input,
                label="Example Tweets — click any to load",
            )

        with gr.Column(scale=3):
            prediction_out = gr.HTML(label="Prediction")
            attribution_out = gr.HTML(label="Token Attributions")
            probs_out = gr.HTML(label="Probability Distribution")

    classify_btn.click(
        fn=predict,
        inputs=tweet_input,
        outputs=[prediction_out, attribution_out, probs_out],
    )

    # Also classify on Enter key in textbox
    tweet_input.submit(
        fn=predict,
        inputs=tweet_input,
        outputs=[prediction_out, attribution_out, probs_out],
    )

    gr.HTML("""
    <div style="text-align:center;padding:20px 0 8px;border-top:1px solid #21262d;
                margin-top:16px;font-family:'Space Mono',monospace;font-size:10px;
                color:#3a4a5a;letter-spacing:1px;">
        B.Tech Final Year Project &nbsp;·&nbsp;
        Models: RoBERTa · DeBERTa · ELECTRA &nbsp;·&nbsp;
        Dataset: HumAID (QCRI) &nbsp;·&nbsp;
        Ensemble Macro F1: 0.769
    </div>
    """)

# Launch with public URL
print("\nLaunching Gradio app...")
print("A public URL will appear below — share it for your demo.\n")
demo.launch(share=True, debug=False)

/tmp/ipykernel_1291/1393068857.py:54: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_1291/1393068857.py:54: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(



Launching Gradio app...
A public URL will appear below — share it for your demo.

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a51b077099c821c417.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
